# 환경 설정

In [48]:
from inspect import FrameInfo
from google.colab import drive
drive.mount('/content/drive')

import os
from io import StringIO
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np
import seaborn as sns
import requests
import xml.etree.ElementTree as ET
import pprint
import warnings
warnings.filterwarnings('ignore')

font_path = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/fonts/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
mpl.rc('font', family = 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 주요 데이터 수집

In [32]:
# 당일 데이터: 실시간으로 추가 중 => 이전 데이터(가장 최신 업데이트) 수집
date = datetime.now() - timedelta(days=2)
date_1 = date.strftime('%Y-%m-%d')
date_2 = date.strftime('%Y%m%d')

### 따릉이 대여/반납 이력 데이터(시간대별, 스테이션별)

In [6]:
api_key = '71474b655273796536324f6f616a61'
all_rows = []  # 수집한 데이터를 담을 빈 리스트
start = 1
end = 1000  # 한 번에 가져올 데이터 양 (서울시 따릉이 API는 한 번의 호출로 최대 1,000건까지 제한적으로 데이터 가져오기 가능)

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/json/tbCycleRentData/{start}/{end}/{date_1}/1/'

    try:
        response = requests.get(url, timeout=10).json()

        # 데이터가 정상적으로 들어있는지 확인
        if 'rentData' in response and 'row' in response['rentData']:
            rows = response['rentData']['row']
            all_rows.extend(rows)  # 가져온 데이터를 리스트에 추가

            print(f"총 {len(all_rows)}건의 데이터")

            # 다음 1000건을 위해 번호 업데이트
            start += 1000
            end += 1000
        else:
            # 더 이상 가져올 데이터가 없으면 반복문 종료
            break

    except Exception as e:
        print(f"오류 발생: {e}")
        break

# 최종 데이터프레임 생성
bike_rent_df = pd.DataFrame(all_rows)
bike_rent_df

총 1000건의 데이터
총 1346건의 데이터


,BIKE_ID,RENT_DT,RENT_ID,RENT_NM,RENT_HOLD,RTN_DT,RTN_ID,RTN_NM,RTN_HOLD,USE_MIN,USE_DST,USR_CLS_CD,SEX_CD,BIRTH_YEAR,RENT_STATION_ID,RETURN_STATION_ID,BIKE_SE_CD,START_INDEX,END_INDEX,RNUM
0,SPB-69944,2026-04-01 01:00:26,01688,마들역 7번출구,0,2026-04-01 01:00:59,01688,마들역 7번출구,0,0,0.00,USR_001,M,1984,ST-1639,ST-1639,일반자전거,0,0,1
1,SPB-45155,2026-04-01 01:00:25,01150,송정역 1번출구,0,2026-04-01 01:01:58,01109,공항시장역 4번출구,0,1,250.00,USR_001,M,1985,ST-1062,ST-511,일반자전거,0,0,2
2,SPB-63341,2026-04-01 01:01:08,02148,낙성대역 3번출구 뒤,0,2026-04-01 01:02:31,02122,낙성대로 입구,0,1,208.60,USR_001,NaN,1962,ST-887,ST-719,일반자전거,0,0,3
3,SPB-84998,2026-04-01 01:01:34,01338,용문2교 옆,99,2026-04-01 01:02:35,01338,용문2교 옆,99,1,0.00,USR_003,NaN,NaN,ST-1202,ST-1202,새싹자전거,0,0,4
4,SPB-80950,2026-04-01 01:02:59,00230,영등포구청역 7번출구,99,2026-04-01 01:03:08,00230,영등포구청역 7번출구,99,0,0.00,USR_001,,1967,ST-413,ST-413,새싹자전거,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1341,SPB-46195,2026-04-01 01:32:13,03575,자양사거리(서원빌딩),0,2026-04-01 07:01:14,NaN,NaN,NaN,60,703.00,USR_001,M,1997,ST-2314,NaN,일반자전거,0,0,1342
1342,SPB-51048,2026-04-01 01:00:50,01101,상사마을 버스정류장(아라대교방면),0,2026-04-01 07:27:47,01200,개화광역환승센터,0,386,15873.23,USR_001,M,1979,ST-507,ST-1695,일반자전거,0,0,1343
1343,SPB-63011,2026-04-01 01:19:23,04128,종암초등학교 뒤(새울센스빌),0,2026-04-01 07:56:12,01380,월곡역 5번출구 앞,0,396,2565.35,USR_001,M,1998,ST-3155,ST-2222,일반자전거,0,0,1344
1344,SPB-56264,2026-04-01 01:58:17,04923,수협대치동지점,0,2026-04-01 10:35:27,03641,대청역,0,517,7049.01,USR_003,NaN,NaN,ST-3207,ST-2867,일반자전거,0,0,1345


In [7]:
bike_rent_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1346 entries, 0 to 1345
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   BIKE_ID            1346 non-null   object
 1   RENT_DT            1346 non-null   object
 2   RENT_ID            1346 non-null   object
 3   RENT_NM            1346 non-null   object
 4   RENT_HOLD          1346 non-null   object
 5   RTN_DT             1346 non-null   object
 6   RTN_ID             1341 non-null   object
 7   RTN_NM             1341 non-null   object
 8   RTN_HOLD           1341 non-null   object
 9   USE_MIN            1346 non-null   object
 10  USE_DST            1346 non-null   object
 11  USR_CLS_CD         1346 non-null   object
 12  SEX_CD             1072 non-null   object
 13  BIRTH_YEAR         1320 non-null   object
 14  RENT_STATION_ID    1346 non-null   object
 15  RETURN_STATION_ID  1341 non-null   object
 16  BIKE_SE_CD         1346 non-null   object


In [8]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'seoul_bike_rent_history.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

bike_rent_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [9]:
# 컬럼명 매핑용 딕셔너리
bike_rent_columns = {
    'BIKE_ID': '자전거ID',
    'RENT_DT': '대여일시',
    'RENT_ID': '대여소번호',
    'RENT_NM': '대여소명',
    'RENT_HOLD': '대여거치대',
    'RTN_DT': '반납일시',
    'RTN_ID': '반납대여소번호',
    'RTN_NM': '반납대여소명',
    'RTN_HOLD': '반납거치대',
    'USE_MIN': '사용시간(분)',
    'USE_DST': '이용거리(M)',
    'USR_CLS_CD': '사용자종류',
    'SEX_CD': '성별',
    'BIRTH_YEAR': '생년',
    'RENT_STATION_ID': '대여대여소ID',
    'RETURN_STATION_ID': '반납대여소ID',
    'BIKE_SE_CD': '자전거구분'
}

### 스테이션 위치 정보 데이터(위도, 경도)

In [10]:
api_key = '706b77416573796534366a566d426a'
all_location_data = []
start = 1
end = 1000 # 한 번에 가져올 데이터 양 (서울시 따릉이 API는 한 번의 호출로 최대 1,000건까지 제한적으로 데이터 가져오기 가능)
list_total_count = None

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/xml/bikeStationMaster/{start}/{end}/'

    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    if list_total_count is None:
        count_element = root.find('list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'
        print(f'총 {list_total_count}건의 데이터')

    # 현재 페이지에서 row 데이터들 찾기
    rows = root.findall('row')

    # 더 이상 가져올 데이터(row)가 없으면 반복 중단
    if not rows:
        break

    # 데이터 파싱하여 리스트에 담기
    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        all_location_data.append(row_dict)

    # 다음 1000건을 위해 번호 이동
    start += 1000
    end += 1000

# 데이터프레임 변환
bike_location_df = pd.DataFrame(all_location_data)
bike_location_df

총 3411건의 데이터


,RNTLS_ID,ADDR1,ADDR2,LAT,LOT
0,ST-10,서울특별시 마포구 양화로 93,427,37.552746,126.918617
1,ST-100,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.536667,127.073593
2,ST-1000,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.510380,126.866798
3,ST-1001,서울특별시 양천구 남부순환로4길20,서서울호수공원,0.000000,0.000000
4,ST-1002,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.529900,126.876541
...,...,...,...,...,...
3406,ST-995,서울특별시 양천구 중앙로 153 공중화장실,None,37.510597,126.857323
3407,ST-996,서울특별시 양천구 남부순환로88길5-16,양강중학교앞 교차로,37.524334,126.850548
3408,ST-997,서울특별시 양천구 목동중앙로 49,목동3단지 시내버스정류장,37.534390,126.869598
3409,ST-998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.000000,0.000000


In [11]:
bike_location_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3411 entries, 0 to 3410
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   RNTLS_ID  3411 non-null   object
 1   ADDR1     3411 non-null   object
 2   ADDR2     1427 non-null   object
 3   LAT       3411 non-null   object
 4   LOT       3411 non-null   object
dtypes: object(5)
memory usage: 133.4+ KB


In [12]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'seoul_bike_location_history.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

bike_location_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [13]:
# 컬럼명 매핑용 딕셔너리
bike_location_columns = {
    'RNTLS_ID': '대여소_ID',
    'ADDR1': '주소1',
    'ADDR2': '주소2',
    'LAT': '위도',
    'LOT': '경도'
}

### 실시간 따릉이 대여/거치 정보 데이터

In [93]:
api_key = '425a554e4f73796536336768454176'
all_rows = []  # 수집한 데이터를 담을 빈 리스트
start = 1
end = 1000  # 한 번에 가져올 데이터 양 (서울시 따릉이 API는 한 번의 호출로 최대 1,000건까지 제한적으로 데이터 가져오기 가능)

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/json/bikeList/{start}/{end}/'

    try:
        response = requests.get(url, timeout=10).json()

        # 데이터가 정상적으로 들어있는지 확인
        if 'realTimeRentData' in response and 'row' in response['realTimeRentData']:
            rows = response['realTimeRentData']['row']
            all_rows.extend(rows)  # 가져온 데이터를 리스트에 추가

            print(f"총 {len(all_rows)}건의 데이터")

            # 다음 1000건을 위해 번호 업데이트
            start += 1000
            end += 1000
        else:
            # 더 이상 가져올 데이터가 없으면 반복문 종료
            break

    except Exception as e:
        print(f"오류 발생: {e}")
        break

# 최종 데이터프레임 생성
real_time_bike_rent_df = pd.DataFrame(all_rows)
real_time_bike_rent_df

""


In [84]:
real_time_bike_rent_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [86]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'real_time_bike_rent.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

real_time_bike_rent_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [87]:
# 컬럼명 매핑용 딕셔너리
real_time_bike_rent_columns ={
    "stationId": "대여소ID",
    "stationName": "대여소명",
    "rackTotCnt": "거치대개수",
    "parkingBikeTotCnt": "자전거주차총건수",
    "shared": "거치율",
    "stationLatitude": "위도",
    "stationLongitude": "경도"
}

# 외부 데이터 수집

### 실시간 미세먼지 데이터

In [21]:
api_key = '706b77416573796534366a566d426a'
real_time_air = []
start = 1
end = 1000 # 한 번에 가져올 데이터 양 (서울시 따릉이 API는 한 번의 호출로 최대 1,000건까지 제한적으로 데이터 가져오기 가능)
list_total_count = None

while True:
    url = f'http://openAPI.seoul.go.kr:8088/{api_key}/xml/ListAirQualityByDistrictService/{start}/{end}/'

    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    if list_total_count is None:
        count_element = root.find('list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'
        print(f'총 {list_total_count}건의 데이터')

    # 현재 페이지에서 row 데이터들 찾기
    rows = root.findall('row')

    # 더 이상 가져올 데이터(row)가 없으면 반복 중단
    if not rows:
        break

    # 데이터 파싱하여 리스트에 담기
    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        real_time_air.append(row_dict)

    # 다음 1000건을 위해 번호 이동
    start += 1000
    end += 1000

# 데이터프레임 변환
real_time_air_df = pd.DataFrame(real_time_air)
real_time_air_df

총 25건의 데이터


,MSRMT_YMD,MSRSTN_PBADMS_CD,MSRSTN_NM,CAI,CAI_GRD,CRST_SBSTN,NTDX,OZON,CBMX,SPDX,PM,FPM
0,202604011600,111123,종로구,79,보통,PM-2.5,0.029,0.05,0.5,0.004,52,39
1,202604011600,111121,중구,82,보통,PM-2.5,0.028,0.042,0.4,0.004,54,36
2,202604011600,111131,용산구,71,보통,PM-10,0.034,0.049,0.5,0.004,62,32
3,202604011600,111142,성동구,71,보통,O3,0.031,0.055,0.4,0.004,57,31
4,202604011600,111141,광진구,74,보통,PM-2.5,0.03,0.039,0.5,0.005,58,36
5,202604011600,111152,동대문구,69,보통,PM-2.5,0.028,0.053,0.4,0.004,53,37
6,202604011600,111151,중랑구,74,보통,PM-2.5,0.029,0.042,0.5,0.004,58,43
7,202604011600,111161,성북구,77,보통,PM-2.5,0.024,0.057,0.6,0.003,60,32
8,202604011600,111291,강북구,78,보통,O3,0.017,0.064,0.4,0.003,51,32
9,202604011600,111171,도봉구,77,보통,O3,0.018,0.063,0.6,0.004,56,38


In [22]:
real_time_air_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   MSRMT_YMD         25 non-null     object
 1   MSRSTN_PBADMS_CD  25 non-null     object
 2   MSRSTN_NM         25 non-null     object
 3   CAI               21 non-null     object
 4   CAI_GRD           21 non-null     object
 5   CRST_SBSTN        21 non-null     object
 6   NTDX              25 non-null     object
 7   OZON              25 non-null     object
 8   CBMX              25 non-null     object
 9   SPDX              25 non-null     object
 10  PM                25 non-null     object
 11  FPM               25 non-null     object
dtypes: object(12)
memory usage: 2.5+ KB


In [23]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'real_time_seoul_air.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

real_time_air_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [24]:
# 컬럼명 매핑용 딕셔너리
real_time_air_columns = {
    "MSRMT_YMD": "측정날짜",
    "MSRSTN_PBADMS_CD": "측정소 행정코드",
    "MSRSTN_NM": "측정소명",
    "CAI": "통합대기환경지수",
    "CAI_GRD": "통합대기환경지수 등급",
    "CRST_SBSTN": "지수결정물질",
    "NTDX": "이산화질소(ppm)",
    "OZON": "오존(ppm)",
    "CBMX": "일산화탄소(ppm)",
    "SPDX": "아황산가스(ppm)",
    "PM": "미세먼지(㎍/㎥)",
    "FPM": "초미세먼지(㎍/㎥)"
}

### 일평균 미세먼지 데이터

In [89]:
api_key = '45516e647473796537345955436648'
daily_avg_air = []
start = 1
end = 1000

start_date_for_daily_air = datetime.strptime('20230101', '%Y%m%d') # 시작
end_date_for_daily_air = datetime.strptime(date_2, '%Y%m%d') # 끝

current_date_loop = start_date_for_daily_air

while current_date_loop <= end_date_for_daily_air:
    date_to_fetch = current_date_loop.strftime('%Y%m%d')
    url = f'http://openAPI.seoul.go.kr:8088/{api_key}/xml/DailyAverageCityAir/{start}/{end}/{date_to_fetch}'

    try:
        response = requests.get(url, timeout=10)
        root = ET.fromstring(response.text)

        rows = root.findall('row')
        if not rows:
            pass
        else:
            for row in rows:
                row_dict = {child.tag: child.text for child in row}
                daily_avg_air.append(row_dict)

    except Exception as e:
        print(f"Error fetching data for {date_to_fetch}: {e}")

    current_date_loop += timedelta(days=1)

# 데이터프레임 변환
daily_avg_air_df = pd.DataFrame(daily_avg_air)
print(f"총 {len(daily_avg_air_df)}건의 데이터 수집 완료.")
daily_avg_air_df

총 29575건의 데이터 수집 완료.


,MSRMT_YMD,SAREA_NM,MSRSTN_NM,PM,FPM,OZON,NTDX,CBMX,SPDX
0,20230101,도심권,중구,57,49,0.025,0.024,0.6,0.004
1,20230101,도심권,종로구,56,45,0.025,0.023,0.7,0.004
2,20230101,도심권,용산구,52,39,0.024,0.02,0.7,0.004
3,20230101,서북권,은평구,58,39,0.023,0.022,0.9,0.004
4,20230101,서북권,서대문구,56,39,0.022,0.025,0.7,0.005
...,...,...,...,...,...,...,...,...,...
29570,20260330,서남권,양천구,55,35,0.0315,0.0412,0.55,0.0033
29571,20260330,동남권,강남구,52,37,0.0374,0.0344,0.51,0.003
29572,20260330,동남권,서초구,61,40,0.0326,0.0331,0.49,0.0032
29573,20260330,동남권,송파구,58,39,0.0325,0.0391,0.5,0.003


In [90]:
daily_avg_air_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29575 entries, 0 to 29574
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   MSRMT_YMD  29575 non-null  object
 1   SAREA_NM   29575 non-null  object
 2   MSRSTN_NM  29575 non-null  object
 3   PM         29575 non-null  object
 4   FPM        29575 non-null  object
 5   OZON       29575 non-null  object
 6   NTDX       29575 non-null  object
 7   CBMX       29575 non-null  object
 8   SPDX       29575 non-null  object
dtypes: object(9)
memory usage: 2.0+ MB


In [91]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'daily_avg_air.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

daily_avg_air_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [92]:
# 컬럼명 매핑용 딕셔너리
daily_avg_air_columns = {
    "MSRMT_YMD": "측정일자",
    "SAREA_NM": "권역명",
    "MSRSTN_NM": "측정소명",
    "PM": "미세먼지(㎍/㎥)",
    "FPM": "초미세먼지(㎍/㎥)",
    "OZON": "오존(ppm)",
    "NTDX": "이산화질소농도(ppm)",
    "CBMX": "일산화탄소농도(ppm)",
    "SPDX": "아황산가스농도(ppm)"
}

### 실시간 기온/기후 통계 데이터

### 일별 기온/기후 통계 데이터

In [76]:
start_date_str = '20230101' # 시작일
start_date_collection = datetime.strptime(start_date_str, '%Y%m%d')
end_date_collection = datetime.strptime(date_2, '%Y%m%d')

all_yearly_data = []
current_date = end_date_collection
start_date_limit = start_date_collection

while current_date >= start_date_limit:
    end_interval = current_date
    start_interval = max(start_date_limit, end_interval - timedelta(days=29)) # 최대 30일까지 불러오기 가능해서, 반복문 사용

    params_yearly = {
        'tm1': start_interval.strftime('%Y%m%d'),    # 시작일
        'tm2': end_interval.strftime('%Y%m%d'),      # 종료일
        'stn_id': '108',        # 서울 지점번호
        'disp': '0',            # 0: 도움말 제외
        'help': '0',            # 0: 도움말 제외
        'authKey': api_key
    }

    response_yearly = requests.get(url, params=params_yearly)

    if response_yearly.status_code == 200:
        full_text_yearly = response_yearly.text

        # 기상청 데이터 특유 주석(#) 및 빈 줄 제거
        data_lines_yearly = [line for line in full_text_yearly.split('\n')
                             if line.strip() and not line.startswith('#')]
        data_body_yearly = '\n'.join(data_lines_yearly)

        # DataFrame 생성 (기존 컬럼명 사용)
        cols = ["YMD", "STN_ID", "LAT", "LON", "ALTD", "TA_DAVG", "TMX_DD", "TMX_OCUR_TMA", "TMN_DD", "TMN_OCUR_TMA", "MRNG_TMN", "MRNG_TMN_OCUR_TMA", "DYTM_TMX", "DYTM_TMX_OCUR_TMA", "NGHT_TMN", "NGHT_TMN_OCUR_TMA", "ETC"]

        df_chunk = pd.read_csv(StringIO(data_body_yearly), names=cols, sep=',', skipinitialspace=True)
        # ETC 컬럼 제거
        df_chunk = df_chunk.drop(columns=['ETC'])
        all_yearly_data.append(df_chunk)
    else:
        print(f"에러 발생: {response_yearly.status_code} - {response_yearly.text}")
        break # 오류 발생 시 루프 종료
    current_date = start_interval - timedelta(days=1) # 다음 구간의 시작 날짜 설정

daily_climatological_df = pd.concat(all_yearly_data, ignore_index=True)
print(f"총 {len(daily_climatological_df)}건의 데이터")

daily_climatological_df

총 1185건의 데이터


,YMD,STN_ID,LAT,LON,ALTD,TA_DAVG,TMX_DD,TMX_OCUR_TMA,TMN_DD,TMN_OCUR_TMA,MRNG_TMN,MRNG_TMN_OCUR_TMA,DYTM_TMX,DYTM_TMX_OCUR_TMA,NGHT_TMN,NGHT_TMN_OCUR_TMA
0,20260301,108,37.57142,126.9658,85.7,9.1,12.8,1525,5.7,717,5.7,717,12.8,1525,5.6,854
1,20260302,108,37.57142,126.9658,85.7,4.9,7.5,1,2.8,2140,5.6,854,5.6,901,2.0,421
2,20260303,108,37.57142,126.9658,85.7,5.9,11.8,1547,2.0,421,2.0,421,11.8,1547,0.7,723
3,20260304,108,37.57142,126.9658,85.7,6.0,12.0,1511,0.7,723,0.7,723,12.0,1511,1.6,555
4,20260305,108,37.57142,126.9658,85.7,6.0,13.2,1341,1.4,2321,1.6,555,13.2,1341,1.4,2321
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1180,20230111,108,37.57142,126.9658,85.7,2.6,9.2,1410,-2.0,802,-2.0,802,9.2,1410,-0.6,645
1181,20230112,108,37.57142,126.9658,85.7,5.9,12.5,1600,-0.6,645,-0.6,645,12.5,1600,5.1,225
1182,20230113,108,37.57142,126.9658,85.7,8.3,10.1,1628,5.1,225,5.1,301,10.1,1628,7.5,819
1183,20230114,108,37.57142,126.9658,85.7,6.6,8.4,1,3.8,2328,7.5,819,8.0,1119,0.8,858


In [77]:
daily_climatological_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1185 entries, 0 to 1184
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   YMD                1185 non-null   int64  
 1   STN_ID             1185 non-null   int64  
 2   LAT                1185 non-null   float64
 3   LON                1185 non-null   float64
 4   ALTD               1185 non-null   float64
 5   TA_DAVG            1185 non-null   float64
 6   TMX_DD             1185 non-null   float64
 7   TMX_OCUR_TMA       1185 non-null   int64  
 8   TMN_DD             1185 non-null   float64
 9   TMN_OCUR_TMA       1185 non-null   int64  
 10  MRNG_TMN           1185 non-null   float64
 11  MRNG_TMN_OCUR_TMA  1185 non-null   int64  
 12  DYTM_TMX           1185 non-null   float64
 13  DYTM_TMX_OCUR_TMA  1185 non-null   int64  
 14  NGHT_TMN           1185 non-null   float64
 15  NGHT_TMN_OCUR_TMA  1185 non-null   int64  
dtypes: float64(9), int64(7)


In [78]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'daily_climatological.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

daily_climatological_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [79]:
daily_climatological_columns = {
    "YMD": "날짜",
    "STN_ID": "지점번호",
    "LAT": "위도",
    "LON": "경도",
    "ALTD": "해발고도",
    "TA_DAVG": "일평균기온",
    "TMX_DD": "일최고기온",
    "TMX_OCUR_TMA": "일최고기온_발생시각",
    "TMN_DD": "일최저기온",
    "TMN_OCUR_TMA": "일최저기온_발생시각",
    "MRNG_TMN": "아침최저기온",
    "MRNG_TMN_OCUR_TMA": "아침최저기온_발생시각",
    "DYTM_TMX": "낮최고기온",
    "DYTM_TMX_OCUR_TMA": "낮최고기온_발생시각",
    "NGHT_TMN": "밤최저기온",
    "NGHT_TMN_OCUR_TMA": "밤최저기온_발생시각"
}

### 실시간 강수량 데이터

### 일별 강수량 데이터

In [115]:
'''
start_date_str = '20230101' # 시작일
start_date_collection = datetime.strptime(start_date_str, '%Y%m%d')
end_date_collection = datetime.strptime(date_2, '%Y%m%d')

all_yearly_data = []
current_date = end_date_collection
start_date_limit = start_date_collection

api_key = 'Squd_oRHS0Wrnf6ER0tFfQ'
url = 'https://apihub.kma.go.kr/api/sts_rn_dd.php'

while current_date >= start_date_limit:
    end_interval = current_date
    start_interval = max(start_date_limit, end_interval - timedelta(days=29)) # 최대 30일까지 불러오기 가능해서, 반복문 사용

    params_yearly = {
        'startDt': start_interval.strftime('%Y%m%d'),
        'endDt': end_interval.strftime('%Y%m%d'),
        'stn_id': '108',
        'authKey': api_key
    }

    response_yearly = requests.get(url, params=params_yearly, timeout=10) # Use the KMA specific URL

    if response_yearly.status_code == 200:
        full_text_yearly = response_yearly.text

        # KMA 데이터는 CSV-like 형식이며, 주석(#) 및 빈 줄 제거
        data_lines_yearly = [line for line in full_text_yearly.split('\n')
                             if line.strip() and not line.startswith('#')]
        data_body_yearly = '\n'.join(data_lines_yearly)

        # DataFrame 생성 (기존 컬럼명 사용)
        cols = ["TMD", "STN_ID", "LAT", "LON", "ALTD", "DST", "RN_DSUM", "RN_MAX_1HR", "RN_MAX_1HR_OCUR_TMA","RN_MAX_6HR", "RN_MAX_6HR_OCUR_TMA","RN_MAX_10M", "RN_MAX_10M_OCUR_TMA"]

        if data_body_yearly:
            df_chunk = pd.read_csv(StringIO(data_body_yearly), names=cols, sep=',', skipinitialspace=True)
            all_yearly_data.append(df_chunk)

    else:
        print(f"에러 발생: {response_yearly.status_code} - {response_yearly.text}")
        break # 오류 발생 시 루프 종료
    current_date = start_interval - timedelta(days=1) # 다음 구간의 시작 날짜 설정

if all_yearly_data:
    daily_precipitation_df = pd.concat(all_yearly_data, ignore_index=True)
    print(f"총 {len(daily_precipitation_df)}건의 데이터")
    display(daily_precipitation_df)
else:
    daily_precipitation_df = pd.DataFrame()
    print("데이터 수집 실패: 유효한 데이터가 없습니다.")
'''

'\nstart_date_str = \'20230101\' # 시작일\nstart_date_collection = datetime.strptime(start_date_str, \'%Y%m%d\')\nend_date_collection = datetime.strptime(date_2, \'%Y%m%d\')\n\nall_yearly_data = []\ncurrent_date = end_date_collection\nstart_date_limit = start_date_collection\n\napi_key = \'Squd_oRHS0Wrnf6ER0tFfQ\'\nurl = \'https://apihub.kma.go.kr/api/sts_rn_dd.php\'\n\nwhile current_date >= start_date_limit:\n    end_interval = current_date\n    start_interval = max(start_date_limit, end_interval - timedelta(days=29)) # 최대 30일까지 불러오기 가능해서, 반복문 사용\n\n    params_yearly = {\n        \'startDt\': start_interval.strftime(\'%Y%m%d\'),\n        \'endDt\': end_interval.strftime(\'%Y%m%d\'),\n        \'stn_id\': \'108\',\n        \'authKey\': api_key\n    }\n\n    response_yearly = requests.get(url, params=params_yearly, timeout=10) # Use the KMA specific URL\n\n    if response_yearly.status_code == 200:\n        full_text_yearly = response_yearly.text\n\n        # KMA 데이터는 CSV-like 형식이며, 주석(#) 및

In [ ]:
daily_precipitation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1185 entries, 0 to 1184
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   YMD                1185 non-null   int64  
 1   STN_ID             1185 non-null   int64  
 2   LAT                1185 non-null   float64
 3   LON                1185 non-null   float64
 4   ALTD               1185 non-null   float64
 5   TA_DAVG            1185 non-null   float64
 6   TMX_DD             1185 non-null   float64
 7   TMX_OCUR_TMA       1185 non-null   int64  
 8   TMN_DD             1185 non-null   float64
 9   TMN_OCUR_TMA       1185 non-null   int64  
 10  MRNG_TMN           1185 non-null   float64
 11  MRNG_TMN_OCUR_TMA  1185 non-null   int64  
 12  DYTM_TMX           1185 non-null   float64
 13  DYTM_TMX_OCUR_TMA  1185 non-null   int64  
 14  NGHT_TMN           1185 non-null   float64
 15  NGHT_TMN_OCUR_TMA  1185 non-null   int64  
dtypes: float64(9), int64(7)


In [ ]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'daily_precipitation.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

daily_precipitation_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [ ]:
daily_precipitation_columns = {
    "TMA": "관측시각",
    "STN_ID": "지점번호",
    "LAT": "위도",
    "LON": "경도",
    "ALTD": "해발고도",
    "RN_DSUM": "일합계강수량",
    "RN_MAX_1HR": "1시간최다강수량",
    "RN_MAX_1HR_OCUR_TMA": "1시간최다강수량발생시각",
    "RN_MAX_6HR": "6시간최다강수량",
    "RN_MAX_6HR_OCUR_TMA": "6시간최다강수량발생시각",
    "RN_MAX_10M": "10분최다강수량",
    "RN_MAX_10M_OCUR_TMA": "10분최다강수량발생시각"
}

### 연령대 및 시간대, 요일별 유동 인구 데이터

In [94]:
api_key = '774c685578737965363648797a6c75'
all_location_data = []
start = 1
end = 1000 # 한 번에 가져올 데이터 양 (서울시 따릉이 API는 한 번의 호출로 최대 1,000건까지 제한적으로 데이터 가져오기 가능)
list_total_count = None

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/xml/VwsmSignguFlpopW/{start}/{end}/'

    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    if list_total_count is None:
        count_element = root.find('list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'
        print(f'총 {list_total_count}건의 데이터')

    # 현재 페이지에서 row 데이터들 찾기
    rows = root.findall('row')

    # 더 이상 가져올 데이터(row)가 없으면 반복 중단
    if not rows:
        break

    # 데이터 파싱하여 리스트에 담기
    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        all_location_data.append(row_dict)

    # 다음 1000건을 위해 번호 이동
    start += 1000
    end += 1000

# 데이터프레임 변환
flow_pop_df = pd.DataFrame(all_location_data)
flow_pop_df

총 700건의 데이터


,STDR_YYQU_CD,SIGNGU_CD,SIGNGU_CD_NM,TOT_FLPOP_CO,ML_FLPOP_CO,FML_FLPOP_CO,AGRDE_10_FLPOP_CO,AGRDE_20_FLPOP_CO,AGRDE_30_FLPOP_CO,AGRDE_40_FLPOP_CO,...,TMZON_14_17_FLPOP_CO,TMZON_17_21_FLPOP_CO,TMZON_21_24_FLPOP_CO,MON_FLPOP_CO,TUES_FLPOP_CO,WED_FLPOP_CO,THUR_FLPOP_CO,FRI_FLPOP_CO,SAT_FLPOP_CO,SUN_FLPOP_CO
0,20244,11410,서대문구,95412427,43271220,52141207,12636065,18857944,15827915,15948911,...,12155647,15497659,11725477,13887907,13978113,13989568,14021133,13902988,12873058,12759658
1,20244,11440,마포구,113763836,51418177,62345659,15937115,23167773,21900941,19405274,...,14616217,20128449,14392166,16017046,16087731,16234139,16249829,16384938,16596707,16193445
2,20244,11200,성동구,73277025,34468611,38808415,9643335,13092408,13464613,12073256,...,9337406,12414692,9035976,10470145,10518016,10575543,10588041,10604166,10316339,10204774
3,20242,11305,강북구,93656261,43035821,50620440,13274486,13240661,12017037,14005656,...,10508420,15227351,12463681,13338443,13184251,13334575,13172644,13161696,13584950,13879704
4,20243,11305,강북구,91978076,42439440,49538635,12737858,12884779,11705176,13819201,...,10434557,15015474,12218433,13026351,13011765,13000983,12962985,12963209,13392134,13620648
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695,20254,11260,중랑구,92797389,43531653,49265736,12709729,12205480,14770452,14203951,...,10197300,15088633,12495305,13112792,12982417,13053988,13080239,13058779,13596256,13912918
696,20254,11380,은평구,93622290,42152012,51470278,15223319,11718339,14057915,15302762,...,9936481,14889849,12746026,13216342,13112955,13186469,13223393,13216181,13713735,13953215
697,20254,11545,금천구,34646869,17161290,17485579,4043862,4831814,6298920,5616799,...,3917897,5563876,4573712,4974705,4928248,4904697,4950811,4915451,4919199,5053759
698,20254,11590,동작구,79515542,37268796,42246747,11537337,15086498,13217054,11972034,...,8789625,12935509,10592534,11248057,11210030,11228825,11325092,11305685,11583443,11614409


In [95]:
flow_pop_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   STDR_YYQU_CD             700 non-null    object
 1   SIGNGU_CD                700 non-null    object
 2   SIGNGU_CD_NM             700 non-null    object
 3   TOT_FLPOP_CO             700 non-null    object
 4   ML_FLPOP_CO              700 non-null    object
 5   FML_FLPOP_CO             700 non-null    object
 6   AGRDE_10_FLPOP_CO        700 non-null    object
 7   AGRDE_20_FLPOP_CO        700 non-null    object
 8   AGRDE_30_FLPOP_CO        700 non-null    object
 9   AGRDE_40_FLPOP_CO        700 non-null    object
 10  AGRDE_50_FLPOP_CO        700 non-null    object
 11  AGRDE_60_ABOVE_FLPOP_CO  700 non-null    object
 12  TMZON_00_06_FLPOP_CO     700 non-null    object
 13  TMZON_06_11_FLPOP_CO     700 non-null    object
 14  TMZON_11_14_FLPOP_CO     700 non-null    o

In [96]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'flow_pop.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

flow_pop_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [97]:
flow_pop_columns = {
"STDR_YYQU_CD": "기준_년분기_코드",
"SIGNGU_CD": "자치구_코드",
"SIGNGU_CD_NM": "자치구_코드_명",
"TOT_FLPOP_CO": "총_유동인구_수",
"ML_FLPOP_CO": "남성_유동인구_수",
"FML_FLPOP_CO": "여성_유동인구_수",
"AGRDE_10_FLPOP_CO": "연령대_10_유동인구_수",
"AGRDE_20_FLPOP_CO": "연령대_20_유동인구_수",
"AGRDE_30_FLPOP_CO": "연령대_30_유동인구_수",
"AGRDE_40_FLPOP_CO": "연령대_40_유동인구_수",
"AGRDE_50_FLPOP_CO": "연령대_50_유동인구_수",
"AGRDE_60_ABOVE_FLPOP_CO": "연령대_60_이상_유동인구_수",
"TMZON_00_06_FLPOP_CO": "시간대_00_06_유동인구_수",
"TMZON_06_11_FLPOP_CO": "시간대_06_11_유동인구_수",
"TMZON_11_14_FLPOP_CO": "시간대_11_14_유동인구_수",
"TMZON_14_17_FLPOP_CO": "시간대_14_17_유동인구_수",
"TMZON_17_21_FLPOP_CO": "시간대_17_21_유동인구_수",
"TMZON_21_24_FLPOP_CO": "시간대_21_24_유동인구_수",
"MON_FLPOP_CO": "월요일_유동인구_수",
"TUES_FLPOP_CO": "화요일_유동인구_수",
"WED_FLPOP_CO": "수요일_유동인구_수",
"THUR_FLPOP_CO": "목요일_유동인구_수",
"FRI_FLPOP_CO": "금요일_유동인구_수",
"SAT_FLPOP_CO": "토요일_유동인구_수",
"SUN_FLPOP_CO": "일요일_유동인구_수"
}

### 자치구 단위 서울 생활인구 데이터

In [98]:
api_key = '774c685578737965363648797a6c75'
all_location_data = []
start = 1
end = 1000 # 한 번에 가져올 데이터 양 (서울시 따릉이 API는 한 번의 호출로 최대 1,000건까지 제한적으로 데이터 가져오기 가능)
list_total_count = None

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/xml/SPOP_LOCAL_RESD_JACHI/{start}/{end}/'

    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    if list_total_count is None:
        count_element = root.find('list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'
        print(f'총 {list_total_count}건의 데이터')

    # 현재 페이지에서 row 데이터들 찾기
    rows = root.findall('row')

    # 더 이상 가져올 데이터(row)가 없으면 반복 중단
    if not rows:
        break

    # 데이터 파싱하여 리스트에 담기
    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        all_location_data.append(row_dict)

    # 다음 1000건을 위해 번호 이동
    start += 1000
    end += 1000

# 데이터프레임 변환
living_pop_df = pd.DataFrame(all_location_data)
living_pop_df

총 1769400건의 데이터


KeyboardInterrupt: 

In [ ]:
living_pop_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   STDR_YYQU_CD             700 non-null    object
 1   SIGNGU_CD                700 non-null    object
 2   SIGNGU_CD_NM             700 non-null    object
 3   TOT_FLPOP_CO             700 non-null    object
 4   ML_FLPOP_CO              700 non-null    object
 5   FML_FLPOP_CO             700 non-null    object
 6   AGRDE_10_FLPOP_CO        700 non-null    object
 7   AGRDE_20_FLPOP_CO        700 non-null    object
 8   AGRDE_30_FLPOP_CO        700 non-null    object
 9   AGRDE_40_FLPOP_CO        700 non-null    object
 10  AGRDE_50_FLPOP_CO        700 non-null    object
 11  AGRDE_60_ABOVE_FLPOP_CO  700 non-null    object
 12  TMZON_00_06_FLPOP_CO     700 non-null    object
 13  TMZON_06_11_FLPOP_CO     700 non-null    object
 14  TMZON_11_14_FLPOP_CO     700 non-null    o

In [ ]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'living_pop.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

living_pop_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [ ]:
living_pop_columns = {
"STDR_DE_ID": "기준일ID",
"TMZON_PD_SE": "시간대구분",
"ADSTRD_CODE_SE": "자치구코드",
"TOT_LVPOP_CO": "총생활인구수",
"MALE_F0T9_LVPOP_CO": "남자0세부터9세생활인구수",
"MALE_F10T14_LVPOP_CO": "남자10세부터14세생활인구수",
"MALE_F15T19_LVPOP_CO": "남자15세부터19세생활인구수",
"MALE_F20T24_LVPOP_CO": "남자20세부터24세생활인구수",
"MALE_F25T29_LVPOP_CO": "남자25세부터29세생활인구수",
"MALE_F30T34_LVPOP_CO": "남자30세부터34세생활인구수",
"MALE_F35T39_LVPOP_CO": "남자35세부터39세생활인구수",
"MALE_F40T44_LVPOP_CO": "남자40세부터44세생활인구수",
"MALE_F45T49_LVPOP_CO": "남자45세부터49세생활인구수",
"MALE_F50T54_LVPOP_CO": "남자50세부터54세생활인구수",
"MALE_F55T59_LVPOP_CO": "남자55세부터59세생활인구수",
"MALE_F60T64_LVPOP_CO": "남자60세부터64세생활인구수",
"MALE_F65T69_LVPOP_CO": "남자65세부터69세생활인구수",
"MALE_F70T74_LVPOP_CO": "남자70세이상생활인구수",
"FEMALE_F0T9_LVPOP_CO": "여자0세부터9세생활인구수",
"FEMALE_F10T14_LVPOP_CO": "여자10세부터14세생활인구수",
"FEMALE_F15T19_LVPOP_CO": "여자15세부터19세생활인구수",
"FEMALE_F20T24_LVPOP_CO": "여자20세부터24세생활인구수",
"FEMALE_F25T29_LVPOP_CO": "여자25세부터29세생활인구수",
"FEMALE_F30T34_LVPOP_CO": "여자30세부터34세생활인구수",
"FEMALE_F35T39_LVPOP_CO": "여자35세부터39세생활인구수",
"FEMALE_F40T44_LVPOP_CO": "여자40세부터44세생활인구수",
"FEMALE_F45T49_LVPOP_CO": "여자45세부터49세생활인구수",
"FEMALE_F50T54_LVPOP_CO": "여자50세부터54세생활인구수",
"FEMALE_F55T59_LVPOP_CO": "여자55세부터59세생활인구수",
"FEMALE_F60T64_LVPOP_CO": "여자60세부터64세생활인구수",
"FEMALE_F65T69_LVPOP_CO": "여자65세부터69세생활인구수",
"FEMALE_F70T74_LVPOP_CO": "여자70세이상생활인구수"
}

# 데이터 전처리

In [ ]:
# 칼럼 매핑
bike_rent_df.rename(columns=bike_rent_columns, inplace=True)
bike_location_df.rename(columns=bike_location_columns, inplace=True)

In [ ]:
bike_rent_df['대여일시'] = pd.to_datetime(bike_rent_df['대여일시'], errors='coerce')
bike_rent_df['반납일시'] = pd.to_datetime(bike_rent_df['반납일시'], errors='coerce')
bike_rent_df['이용거리(M)'] = pd.to_numeric(bike_rent_df['이용거리(M)'], errors='coerce')
bike_rent_df['사용시간(분)'] = pd.to_numeric(bike_rent_df['사용시간(분)'], errors='coerce')

bike_rent_df.info()

bike_location_df['위도'] = pd.to_numeric(bike_location_df['위도'], errors='coerce')
bike_location_df['경도'] = pd.to_numeric(bike_location_df['경도'], errors='coerce')
bike_location_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 968 entries, 0 to 967
Data columns (total 20 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   자전거ID        968 non-null    object        
 1   대여일시         968 non-null    datetime64[ns]
 2   대여소번호        968 non-null    object        
 3   대여소명         968 non-null    object        
 4   대여거치대        968 non-null    object        
 5   반납일시         968 non-null    datetime64[ns]
 6   반납대여소번호      962 non-null    object        
 7   반납대여소명       962 non-null    object        
 8   반납거치대        962 non-null    object        
 9   사용시간(분)      968 non-null    int64         
 10  이용거리(M)      968 non-null    float64       
 11  사용자종류        968 non-null    object        
 12  성별           774 non-null    object        
 13  생년           939 non-null    object        
 14  대여대여소ID      968 non-null    object        
 15  반납대여소ID      962 non-null    object        
 16  자전거구분   